In [5]:
# ============================================================
# 14_naked_atm_put_tenor_sizing_v0_1
# Naked ATM put tenor + sizing research
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

cwd = Path.cwd()

# Force project root if notebook is launched from /notebooks
if cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
AUDIT_DIR = PROJECT_ROOT / "data" / "audit"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

assert PROCESSED_DATA_DIR.exists(), PROCESSED_DATA_DIR
assert AUDIT_DIR.exists(), AUDIT_DIR

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("cwd:", cwd)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("AUDIT_DIR:", AUDIT_DIR)

cwd: C:\Users\patri\vrp_project\notebooks
PROJECT_ROOT: C:\Users\patri\vrp_project
PROCESSED_DATA_DIR: C:\Users\patri\vrp_project\data\processed
AUDIT_DIR: C:\Users\patri\vrp_project\data\audit


In [6]:
# ============================================================
# Scope
# ============================================================

NOTEBOOK_SCOPE = {
    "instrument": "naked ATM puts",
    "strike_rule": "ATM only",
    "varying_dimensions": ["tenor", "sizing"],
    "tenors": [9, 12, 15, 18, 21, 24, 27, 30, 33],
    "excluded_for_now": [
        "strike optimization",
        "spreads",
        "hedges",
        "stress engine",
        "margin usage",
        "portfolio caps",
        "vol-of-vol overlays",
        "new signal mining",
    ],
}

NOTEBOOK_SCOPE

{'instrument': 'naked ATM puts',
 'strike_rule': 'ATM only',
 'varying_dimensions': ['tenor', 'sizing'],
 'tenors': [9, 12, 15, 18, 21, 24, 27, 30, 33],
 'excluded_for_now': ['strike optimization',
  'spreads',
  'hedges',
  'stress engine',
  'margin usage',
  'portfolio caps',
  'vol-of-vol overlays',
  'new signal mining']}

In [7]:
# ============================================================
# Load frozen signal selections from prior research
# ============================================================

COMBO_SELECTED_TRADES_PARQUET = (
    PROCESSED_DATA_DIR / "put_shortlist_sleeve_combo_selected_trades_v0_2.parquet"
)

REFINED_CANDIDATE_PARQUET = (
    PROCESSED_DATA_DIR / "put_refined_hybrid_override_selected_trades_v0_2.parquet"
)

assert COMBO_SELECTED_TRADES_PARQUET.exists(), COMBO_SELECTED_TRADES_PARQUET
assert REFINED_CANDIDATE_PARQUET.exists(), REFINED_CANDIDATE_PARQUET

combo_df = pd.read_parquet(COMBO_SELECTED_TRADES_PARQUET).copy()
candidate_df = pd.read_parquet(REFINED_CANDIDATE_PARQUET).copy()

combo_df["trade_dt"] = pd.to_datetime(combo_df["trade_dt"])
candidate_df["trade_dt"] = pd.to_datetime(candidate_df["trade_dt"])

combo_df["entry_tenor"] = combo_df["entry_tenor"].astype(int)
candidate_df["entry_tenor"] = candidate_df["entry_tenor"].astype(int)

primary_only = (
    combo_df
    .query("combo_strategy == 'primary_only'")
    .copy()
    .sort_values("trade_dt")
    .reset_index(drop=True)
)

candidate = (
    candidate_df
    .copy()
    .sort_values("trade_dt")
    .reset_index(drop=True)
)

candidate["combo_strategy"] = "primary_with_refined_hybrid_override"

print("Primary rows:", len(primary_only))
print("Candidate rows:", len(candidate))

display(primary_only.head())
display(candidate.head())

Primary rows: 602
Candidate rows: 602


,trade_date,tenor,spx_close,spx_rsi_14,vix_style_vol,implied_variance,old_vrp_signal,old_vol,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_forecast_model,simple_forecast_variance,forecast_vol,forecast_vrp_signal,forecast_vrp_3m_z,forecast_vrp_1y_z,forecast_vrp_blended_z,target_tenor,side,source_universe,trade_date_ts,target_expiry_date,entry_spx_close,tenor_group,trade_year,expiry_mapping_status,selected_expiry_date,selected_chain_file,expiry_distance_days,actual_dte,strike_mapping_status,short_strike_selected,raw_short_strike,put_moneyness,put_pct_otm,entry_quote_status,entry_bid,entry_ask,entry_mid,entry_spread,entry_spread_pct_mid,mapping_status,expiry_spx_close,expiry_spx_status,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,dollars_per_contract,contracts_for_normalized_notional,normalized_pnl_dollars,normalized_pnl_pct,win,pricing_status,trade_dt,entry_tenor,tenor_champion_panel,champion_forecast_vol,champion_forecast_vrp_signal,champion_forecast_vrp_signal_z_3m,champion_forecast_vrp_signal_z_1y,champion_forecast_vrp_min_z,simple_carry_vrp_signal,trailing_carry_vrp_signal,entry_rsi_14,test_pnl,test_pnl_source_col,test_win,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,simple_minus_champion_signal,both_simple_and_champion_rich_flag,simple_rich_champion_weak_flag,champion_rich_simple_weak_flag,tenor_denom,vrp_signal_har_rv_iv,vrp_signal_har_rv_iv_min_z,pred_vol_pct_har_rv_iv,vrp_signal_har_rv_iv_per_tenor,vrp_signal_har_rv_iv_per_tenor_min_z,pred_vol_pct_har_rv_iv_per_tenor,vrp_signal_hybrid_log_objective,vrp_signal_hybrid_log_objective_min_z,pred_vol_pct_hybrid_log_objective,vrp_signal_hybrid_back_har_only,vrp_signal_hybrid_back_har_only_min_z,pred_vol_pct_hybrid_back_har_only,score,rule_id,rule_name,denominator_model,shortlist_label,rsi_cap,old_signal_threshold,old_z_threshold,simple_signal_threshold,simple_z_threshold,denom_signal_threshold,denom_z_threshold,priority,combo_strategy
0,20201029,15,3310.11,40.226305,40.848074,0.166857,1.168026,22.779177,0.621002,0.893562,0.757282,ewma_10_scheduled_bias_adjusted,0.039545,19.886021,1.439685,1.556606,1.333023,1.444815,15,put,all_date_tenor_atm_put,2020-10-29,20201113,3310.11,front_9_15d,2020,mapped,20201113,C:\Users\patri\vrp_project\data\raw\thetadata_...,0,15,mapped,3310.0,3310.0,0.999967,0.000033,mapped,91.9,107.9,99.90,16.0,0.160160,mapped,3585.15,mapped,0.0,True,91.9,91.9,9190.0,3.021048,27763.427801,0.027763,True,priced,2020-10-29,15,15.0,38.653299,0.110455,-0.962036,-1.143295,-1.143295,1.439685,1.168026,40.226305,27763.427801,normalized_pnl_dollars,True,1.498513,1.321883,1.321883,0.617793,0.893814,0.617793,1.329230,True,False,False,15.0,0.110455,-1.143295,38.653299,0.231254,-0.889581,36.387771,0.084361,-1.253312,39.160920,0.084361,-1.253312,39.160920,1.476922,1729,old_simple_confirm_reference,none_reference,primary_old_simple_robust,72,0.5,-0.5,0.4,0.0,NaN,NaN,1,primary_only
1,20201030,15,3269.96,37.052248,40.741627,0.165988,1.096323,23.549131,0.496804,0.790689,0.643746,ewma_10_scheduled_bias_adjusted,0.035423,18.821078,1.544546,1.785395,1.477190,1.631293,15,put,all_date_tenor_atm_put,2020-10-30,20201114,3269.96,front_9_15d,2020,mapped,20201113,C:\Users\patri\vrp_project\data\raw\thetadata_...,-1,14,mapped,3265.0,3269.0,0.998483,0.001517,mapped,89.9,96.8,93.35,6.9,0.073915,mapped,3585.15,mapped,0.0,True,89.9,89.9,8990.0,3.058141,27492.691042,0.027493,True,priced,2020-10-30,15,15.0,30.773523,0.561191,0.446172,0.087274,0.087274,1.544546,1.096323,37.052248,27492.691042,normalized_pnl_dollars,True,1.710180,1.464655,1.464655,0.496463,0.792999,0.496463,0.983355,True,False,False,15.0,0.561191,0.087274,30.773523,0.023412,-1.763964,40.267483,0.195515,-0.936773,36.947301,0.195515,-0.936773,36.947301,1.344555,1729,old_simple_confirm_reference,none_reference,primary_old_simple_robust,72,0.5,-0.5,0.4,0.0,NaN,NaN,1,primary_only


,trade_date,tenor,spx_close,spx_rsi_14,vix_style_vol,implied_variance,old_vrp_signal,old_vol,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_forecast_model,simple_forecast_variance,forecast_vol,forecast_vrp_signal,forecast_vrp_3m_z,forecast_vrp_1y_z,forecast_vrp_blended_z,target_tenor,side,source_universe,trade_date_ts,target_expiry_date,entry_spx_close,tenor_group,trade_year,expiry_mapping_status,selected_expiry_date,selected_chain_file,expiry_distance_days,actual_dte,strike_mapping_status,short_strike_selected,raw_short_strike,put_moneyness,put_pct_otm,entry_quote_status,entry_bid,entry_ask,entry_mid,entry_spread,entry_spread_pct_mid,mapping_status,expiry_spx_close,expiry_spx_status,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,dollars_per_contract,contracts_for_normalized_notional,normalized_pnl_dollars,normalized_pnl_pct,win,pricing_status,trade_dt,entry_tenor,tenor_champion_panel,champion_forecast_vol,champion_forecast_vrp_signal,champion_forecast_vrp_signal_z_3m,champion_forecast_vrp_signal_z_1y,champion_forecast_vrp_min_z,simple_carry_vrp_signal,trailing_carry_vrp_signal,entry_rsi_14,test_pnl,test_pnl_source_col,test_win,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,simple_minus_champion_signal,both_simple_and_champion_rich_flag,simple_rich_champion_weak_flag,champion_rich_simple_weak_flag,tenor_denom,vrp_signal_har_rv_iv,vrp_signal_har_rv_iv_min_z,pred_vol_pct_har_rv_iv,vrp_signal_har_rv_iv_per_tenor,vrp_signal_har_rv_iv_per_tenor_min_z,pred_vol_pct_har_rv_iv_per_tenor,vrp_signal_hybrid_log_objective,vrp_signal_hybrid_log_objective_min_z,pred_vol_pct_hybrid_log_objective,vrp_signal_hybrid_back_har_only,vrp_signal_hybrid_back_har_only_min_z,pred_vol_pct_hybrid_back_har_only,score,rule_id,rule_name,denominator_model,shortlist_label,rsi_cap,old_signal_threshold,old_z_threshold,simple_signal_threshold,simple_z_threshold,denom_signal_threshold,denom_z_threshold,priority,combo_strategy
0,20201029,15,3310.11,40.226305,40.848074,0.166857,1.168026,22.779177,0.621002,0.893562,0.757282,ewma_10_scheduled_bias_adjusted,0.039545,19.886021,1.439685,1.556606,1.333023,1.444815,15,put,all_date_tenor_atm_put,2020-10-29,20201113,3310.11,front_9_15d,2020,mapped,20201113,C:\Users\patri\vrp_project\data\raw\thetadata_...,0,15,mapped,3310.0,3310.0,0.999967,0.000033,mapped,91.9,107.9,99.90,16.0,0.160160,mapped,3585.15,mapped,0.0,True,91.9,91.9,9190.0,3.021048,27763.427801,0.027763,True,priced,2020-10-29,15,15.0,38.653299,0.110455,-0.962036,-1.143295,-1.143295,1.439685,1.168026,40.226305,27763.427801,normalized_pnl_dollars,True,1.498513,1.321883,1.321883,0.617793,0.893814,0.617793,1.329230,True,False,False,15.0,0.110455,-1.143295,38.653299,0.231254,-0.889581,36.387771,0.084361,-1.253312,39.160920,0.084361,-1.253312,39.160920,1.476922,1729.0,old_simple_confirm_reference,none_reference,primary_old_simple_robust,72,0.5,-0.5,0.4,0.0,NaN,NaN,1,primary_with_refined_hybrid_override
1,20201030,15,3269.96,37.052248,40.741627,0.165988,1.096323,23.549131,0.496804,0.790689,0.643746,ewma_10_scheduled_bias_adjusted,0.035423,18.821078,1.544546,1.785395,1.477190,1.631293,15,put,all_date_tenor_atm_put,2020-10-30,20201114,3269.96,front_9_15d,2020,mapped,20201113,C:\Users\patri\vrp_project\data\raw\thetadata_...,-1,14,mapped,3265.0,3269.0,0.998483,0.001517,mapped,89.9,96.8,93.35,6.9,0.073915,mapped,3585.15,mapped,0.0,True,89.9,89.9,8990.0,3.058141,27492.691042,0.027493,True,priced,2020-10-30,15,15.0,30.773523,0.561191,0.446172,0.087274,0.087274,1.544546,1.096323,37.052248,27492.691042,normalized_pnl_dollars,True,1.710180,1.464655,1.464655,0.496463,0.792999,0.496463,0.983355,True,False,False,15.0,0.561191,0.087274,30.773523,0.023412,-1.763964,40.267483,0.195515,-0.936773,36.947301,0.195515,-0.936773,36.947301,1.344555,1729.0,old_simple_confirm_reference,none_reference,primary_old_simple_robust,72,0.5,-0.5,0.4

In [8]:
# ============================================================
# Column sanity check
# ============================================================

print("Primary columns:")
print(primary_only.columns.tolist())

print("\nCandidate columns:")
print(candidate.columns.tolist())

required_cols = [
    "trade_dt",
    "entry_tenor",
    "tenor_group",
    "test_pnl",
]

for c in required_cols:
    assert c in primary_only.columns, f"Missing from primary_only: {c}"
    assert c in candidate.columns, f"Missing from candidate: {c}"

print("\nRequired columns present.")

Primary columns:
['trade_date', 'tenor', 'spx_close', 'spx_rsi_14', 'vix_style_vol', 'implied_variance', 'old_vrp_signal', 'old_vol', 'old_vrp_3m_z', 'old_vrp_1y_z', 'old_vrp_blended_z', 'simple_forecast_model', 'simple_forecast_variance', 'forecast_vol', 'forecast_vrp_signal', 'forecast_vrp_3m_z', 'forecast_vrp_1y_z', 'forecast_vrp_blended_z', 'target_tenor', 'side', 'source_universe', 'trade_date_ts', 'target_expiry_date', 'entry_spx_close', 'tenor_group', 'trade_year', 'expiry_mapping_status', 'selected_expiry_date', 'selected_chain_file', 'expiry_distance_days', 'actual_dte', 'strike_mapping_status', 'short_strike_selected', 'raw_short_strike', 'put_moneyness', 'put_pct_otm', 'entry_quote_status', 'entry_bid', 'entry_ask', 'entry_mid', 'entry_spread', 'entry_spread_pct_mid', 'mapping_status', 'expiry_spx_close', 'expiry_spx_status', 'expiry_intrinsic_value', 'expired_otm', 'entry_credit_points', 'short_option_pnl_points', 'dollars_per_contract', 'contracts_for_normalized_notional',

In [9]:
# ============================================================
# Unsized baseline comparison
# ============================================================

def max_drawdown_from_pnl(pnl_series):
    pnl = pd.Series(pnl_series).fillna(0.0)
    equity = pnl.cumsum()
    running_max = equity.cummax()
    dd = equity - running_max
    return float(dd.min())

def summarize_unsized(df, strategy_name):
    df = df.sort_values("trade_dt").copy()
    df["trade_year"] = df["trade_dt"].dt.year

    pnl = df["test_pnl"].astype(float)
    yearly = df.groupby("trade_year")["test_pnl"].sum()

    max_dd = max_drawdown_from_pnl(pnl)

    return {
        "strategy_name": strategy_name,
        "trades": len(df),
        "start_date": df["trade_dt"].min(),
        "end_date": df["trade_dt"].max(),
        "total_pnl": pnl.sum(),
        "avg_pnl": pnl.mean(),
        "median_pnl": pnl.median(),
        "win_rate": (pnl > 0).mean(),
        "q05_pnl": pnl.quantile(0.05),
        "worst_pnl": pnl.min(),
        "max_drawdown": max_dd,
        "pnl_to_drawdown": pnl.sum() / abs(max_dd) if max_dd != 0 else np.nan,
        "worst_10_trade_sum": pnl.rolling(10).sum().min(),
        "worst_20_trade_sum": pnl.rolling(20).sum().min(),
        "positive_year_rate": (yearly > 0).mean(),
        "min_year_pnl": yearly.min(),
        "avg_tenor": df["entry_tenor"].mean(),
        "front_pct": df["tenor_group"].eq("front_9_15d").mean(),
        "middle_pct": df["tenor_group"].eq("middle_18_24d").mean(),
        "back_pct": df["tenor_group"].eq("back_27_33d").mean(),
    }

unsized_summary = pd.DataFrame([
    summarize_unsized(primary_only, "primary_only"),
    summarize_unsized(candidate, "primary_with_refined_hybrid_override"),
])

display(unsized_summary)

,strategy_name,trades,start_date,end_date,total_pnl,avg_pnl,median_pnl,win_rate,q05_pnl,worst_pnl,max_drawdown,pnl_to_drawdown,worst_10_trade_sum,worst_20_trade_sum,positive_year_rate,min_year_pnl,avg_tenor,front_pct,middle_pct,back_pct
0,primary_only,602,2020-10-29,2026-06-05,3.544466e+06,5887.817014,10442.162535,0.779070,-27785.830930,-111511.425505,-944746.911842,3.751762,-533576.106123,-941213.622876,1.0,102416.500948,21.019934,0.392027,0.224252,0.383721
1,primary_with_refined_hybrid_override,602,2020-10-29,2026-06-05,3.795287e+06,6304.463174,10937.260361,0.782392,-26231.923947,-111511.425505,-944746.911842,4.017252,-533576.106123,-941213.622876,1.0,112187.213271,21.996678,0.352159,0.186047,0.461794


In [16]:
# ============================================================
# Cells 6-9 consolidated:
# Naked ATM put sizing comparison
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Ensure tenor_group exists
# ------------------------------------------------------------

def infer_tenor_group(tenor):
    tenor = int(tenor)
    if tenor <= 15:
        return "front_9_15d"
    elif tenor <= 24:
        return "middle_18_24d"
    else:
        return "back_27_33d"

for df in [primary_only, candidate]:
    df["trade_dt"] = pd.to_datetime(df["trade_dt"])
    df["entry_tenor"] = df["entry_tenor"].astype(int)
    if "tenor_group" not in df.columns:
        df["tenor_group"] = df["entry_tenor"].map(infer_tenor_group)

# ------------------------------------------------------------
# Build primary and candidate sizing frames
# ------------------------------------------------------------

primary_sizing = primary_only.copy()
primary_sizing["is_refined_override"] = False

candidate_sizing = candidate.copy()

primary_lookup = (
    primary_only
    .set_index("trade_dt")[["entry_tenor", "tenor_group"]]
    .rename(columns={
        "entry_tenor": "primary_entry_tenor",
        "tenor_group": "primary_tenor_group",
    })
)

candidate_sizing = candidate_sizing.merge(
    primary_lookup,
    left_on="trade_dt",
    right_index=True,
    how="left",
)

candidate_sizing["is_refined_override"] = (
    candidate_sizing["entry_tenor"] != candidate_sizing["primary_entry_tenor"]
)

print("Primary trades:", len(primary_sizing))
print("Candidate trades:", len(candidate_sizing))
print("Refined override trades:", int(candidate_sizing["is_refined_override"].sum()))

display(
    candidate_sizing
    .groupby(["is_refined_override", "tenor_group"])
    .size()
    .reset_index(name="trades")
)

# ------------------------------------------------------------
# Define sizing schemes
# ------------------------------------------------------------

SIZING_SCHEMES = {
    "equal_1x": {
        "front_9_15d": 1.00,
        "middle_18_24d": 1.00,
        "back_27_33d": 1.00,
        "override_size": None,
    },
    "tenor_based": {
        "front_9_15d": 0.75,
        "middle_18_24d": 1.00,
        "back_27_33d": 1.15,
        "override_size": None,
    },
    "tenor_plus_override_sizeup": {
        "front_9_15d": 0.75,
        "middle_18_24d": 1.00,
        "back_27_33d": 1.15,
        "override_size": 1.25,
    },
}

def assign_size_multiplier(df, scheme_name):
    scheme = SIZING_SCHEMES[scheme_name]

    size = df["tenor_group"].map({
        "front_9_15d": scheme["front_9_15d"],
        "middle_18_24d": scheme["middle_18_24d"],
        "back_27_33d": scheme["back_27_33d"],
    }).astype(float)

    if scheme["override_size"] is not None:
        size = size.where(~df["is_refined_override"], scheme["override_size"])

    return size

# ------------------------------------------------------------
# Summary helpers
# ------------------------------------------------------------

def max_drawdown_from_pnl(pnl_series):
    pnl = pd.Series(pnl_series).fillna(0.0)
    equity = pnl.cumsum()
    running_max = equity.cummax()
    dd = equity - running_max
    return float(dd.min())

def summarize_sized(df, strategy_name, sizing_scheme):
    df = df.sort_values("trade_dt").copy()
    df["trade_year"] = df["trade_dt"].dt.year

    pnl = df["sized_pnl"].astype(float)
    yearly = df.groupby("trade_year")["sized_pnl"].sum()

    max_dd = max_drawdown_from_pnl(pnl)

    total_size_units = df["size_multiplier"].sum()
    avg_size_multiplier = df["size_multiplier"].mean()

    return {
        "strategy_name": strategy_name,
        "sizing_scheme": sizing_scheme,
        "trades": len(df),
        "start_date": df["trade_dt"].min(),
        "end_date": df["trade_dt"].max(),
        "total_pnl": pnl.sum(),
        "avg_pnl": pnl.mean(),
        "median_pnl": pnl.median(),
        "win_rate": (pnl > 0).mean(),
        "q05_pnl": pnl.quantile(0.05),
        "worst_pnl": pnl.min(),
        "max_drawdown": max_dd,
        "pnl_to_drawdown": pnl.sum() / abs(max_dd) if max_dd != 0 else np.nan,
        "worst_10_trade_sum": pnl.rolling(10).sum().min(),
        "worst_20_trade_sum": pnl.rolling(20).sum().min(),
        "positive_year_rate": (yearly > 0).mean(),
        "min_year_pnl": yearly.min(),
        "avg_tenor": df["entry_tenor"].mean(),
        "front_pct": df["tenor_group"].eq("front_9_15d").mean(),
        "middle_pct": df["tenor_group"].eq("middle_18_24d").mean(),
        "back_pct": df["tenor_group"].eq("back_27_33d").mean(),
        "avg_size_multiplier": avg_size_multiplier,
        "total_size_units": total_size_units,
        "pnl_per_size_unit": pnl.sum() / total_size_units if total_size_units != 0 else np.nan,
    }

# ------------------------------------------------------------
# Run sizing comparison
# ------------------------------------------------------------

base_frames = {
    "primary_only": primary_sizing,
    "primary_with_refined_hybrid_override": candidate_sizing,
}

summary_rows = []
sized_trade_frames = []

for strategy_name, base_df in base_frames.items():
    for sizing_scheme in SIZING_SCHEMES.keys():
        temp = base_df.copy()

        temp["strategy_name"] = strategy_name
        temp["sizing_scheme"] = sizing_scheme
        temp["size_multiplier"] = assign_size_multiplier(temp, sizing_scheme)
        temp["sized_pnl"] = temp["test_pnl"] * temp["size_multiplier"]

        summary_rows.append(
            summarize_sized(temp, strategy_name, sizing_scheme)
        )

        sized_trade_frames.append(temp)

sizing_summary = pd.DataFrame(summary_rows)
sized_trades = pd.concat(sized_trade_frames, ignore_index=True)

# Save
SIZING_SWEEP_CSV = AUDIT_DIR / "naked_atm_put_tenor_sizing_sweep_v0_1.csv"
SIZED_TRADES_PARQUET = PROCESSED_DATA_DIR / "naked_atm_put_tenor_sizing_trades_v0_1.parquet"

sizing_summary.to_csv(SIZING_SWEEP_CSV, index=False)
sized_trades.to_parquet(SIZED_TRADES_PARQUET, index=False)

print("Saved:")
print(SIZING_SWEEP_CSV)
print(SIZED_TRADES_PARQUET)

print("\nSizing summary:")
display(
    sizing_summary
    .sort_values(["strategy_name", "sizing_scheme"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Candidate by year and tenor
# ------------------------------------------------------------

candidate_sized = sized_trades[
    sized_trades["strategy_name"].eq("primary_with_refined_hybrid_override")
].copy()

candidate_sized["trade_year"] = candidate_sized["trade_dt"].dt.year

candidate_by_year = (
    candidate_sized
    .groupby(["sizing_scheme", "trade_year"])
    .agg(
        trades=("trade_dt", "count"),
        total_pnl=("sized_pnl", "sum"),
        avg_pnl=("sized_pnl", "mean"),
        win_rate=("sized_pnl", lambda x: (x > 0).mean()),
        worst_pnl=("sized_pnl", "min"),
        avg_size_multiplier=("size_multiplier", "mean"),
        avg_tenor=("entry_tenor", "mean"),
        front_pct=("tenor_group", lambda x: (x == "front_9_15d").mean()),
        middle_pct=("tenor_group", lambda x: (x == "middle_18_24d").mean()),
        back_pct=("tenor_group", lambda x: (x == "back_27_33d").mean()),
    )
    .reset_index()
)

agg_dict = {
    "trades": ("trade_dt", "count"),
    "total_pnl": ("sized_pnl", "sum"),
    "avg_pnl": ("sized_pnl", "mean"),
    "win_rate": ("sized_pnl", lambda x: (x > 0).mean()),
    "worst_pnl": ("sized_pnl", "min"),
    "avg_size_multiplier": ("size_multiplier", "mean"),
}

if "entry_rsi_14" in candidate_sized.columns:
    agg_dict["avg_rsi"] = ("entry_rsi_14", "mean")

candidate_by_tenor = (
    candidate_sized
    .groupby(["sizing_scheme", "entry_tenor"])
    .agg(**agg_dict)
    .reset_index()
)

print("\nCandidate by year:")
display(candidate_by_year)

print("\nCandidate by tenor:")
display(candidate_by_tenor)

Primary trades: 602
Candidate trades: 602
Refined override trades: 47


,is_refined_override,tenor_group,trades
0,False,back_27_33d,231
1,False,front_9_15d,212
2,False,middle_18_24d,112
3,True,back_27_33d,47


Saved:
C:\Users\patri\vrp_project\data\audit\naked_atm_put_tenor_sizing_sweep_v0_1.csv
C:\Users\patri\vrp_project\data\processed\naked_atm_put_tenor_sizing_trades_v0_1.parquet

Sizing summary:


,strategy_name,sizing_scheme,trades,start_date,end_date,total_pnl,avg_pnl,median_pnl,win_rate,q05_pnl,worst_pnl,max_drawdown,pnl_to_drawdown,worst_10_trade_sum,worst_20_trade_sum,positive_year_rate,min_year_pnl,avg_tenor,front_pct,middle_pct,back_pct,avg_size_multiplier,total_size_units,pnl_per_size_unit
0,primary_only,equal_1x,602,2020-10-29,2026-06-05,3.544466e+06,5887.817014,10442.162535,0.779070,-27785.830930,-111511.425505,-9.447469e+05,3.751762,-533576.106123,-941213.622876,1.0,102416.500948,21.019934,0.392027,0.224252,0.383721,1.000000,602.00,5887.817014
1,primary_only,tenor_based,602,2020-10-29,2026-06-05,3.660791e+06,6081.048410,8850.699234,0.779070,-24969.633713,-128238.139331,-1.013815e+06,3.610906,-586123.120605,-994096.652684,1.0,83869.648508,21.019934,0.392027,0.224252,0.383721,0.959551,577.65,6337.386208
2,primary_only,tenor_plus_override_sizeup,602,2020-10-29,2026-06-05,3.660791e+06,6081.048410,8850.699234,0.779070,-24969.633713,-128238.139331,-1.013815e+06,3.610906,-586123.120605,-994096.652684,1.0,83869.648508,21.019934,0.392027,0.224252,0.383721,0.959551,577.65,6337.386208
3,primary_with_refined_hybrid_override,equal_1x,602,2020-10-29,2026-06-05,3.795287e+06,6304.463174,10937.260361,0.782392,-26231.923947,-111511.425505,-9.447469e+05,4.017252,-533576.106123,-941213.622876,1.0,112187.213271,21.996678,0.352159,0.186047,0.461794,1.000000,602.00,6304.463174
4,primary_with_refined_hybrid_override,tenor_based,602,2020-10-29,2026-06-05,3.986522e+06,6622.130393,9172.095359,0.782392,-22496.162449,-128238.139331,-1.013815e+06,3.932198,-586123.120605,-994096.652684,1.0,100464.806389,21.996678,0.352159,0.186047,0.461794,0.981229,590.70,6748.810728
5,primary_with_refined_hybrid_override,tenor_plus_override_sizeup,602,2020-10-29,2026-06-05,4.030386e+06,6694.992835,9172.095359,0.782392,-22496.162449,-128238.139331,-1.013815e+06,3.975464,-586123.120605,-994096.652684,1.0,103638.230683,21.996678,0.352159,0.186047,0.461794,0.989037,595.40,6769.206729



Candidate by year:


,sizing_scheme,trade_year,trades,total_pnl,avg_pnl,win_rate,worst_pnl,avg_size_multiplier,avg_tenor,front_pct,middle_pct,back_pct
0,equal_1x,2020,22,3.043181e+05,13832.640413,0.954545,-1389.519670,1.000000,13.090909,0.909091,0.090909,0.000000
1,equal_1x,2021,137,1.292520e+06,9434.451512,0.802920,-26652.128198,1.000000,24.700730,0.189781,0.226277,0.583942
2,equal_1x,2022,36,2.951481e+05,8198.558882,0.694444,-40497.519097,1.000000,15.333333,0.722222,0.111111,0.166667
3,equal_1x,2023,134,7.877937e+05,5879.057740,0.761194,-49261.410409,1.000000,21.470149,0.380597,0.194030,0.425373
4,equal_1x,2024,112,7.883131e+05,7038.509449,0.848214,-38313.419003,1.000000,20.571429,0.401786,0.205357,0.392857
5,equal_1x,2025,106,1.121872e+05,1058.369937,0.792453,-111511.425505,1.000000,23.037736,0.320755,0.179245,0.500000
6,equal_1x,2026,55,2.150068e+05,3909.213745,0.618182,-57319.352129,1.000000,25.363636,0.181818,0.127273,0.690909
7,tenor_based,2020,22,2.350843e+05,10685.651166,0.954545,-1042.139752,0.772727,13.090909,0.909091,0.090909,0.000000
8,tenor_based,2021,137,1.436576e+06,10485.954285,0.802920,-27646.222776,1.040146,24.700730,0.189781,0.226277,0.583942
9,tenor_based,2022,36,2.884917e+05,8013.658830,0.694444,-40497.519097,0.844444,15.333333,0.722222,0.111111,0.166667



Candidate by tenor:


,sizing_scheme,entry_tenor,trades,total_pnl,avg_pnl,win_rate,worst_pnl,avg_size_multiplier,avg_rsi
0,equal_1x,9,58,1.075213e+05,1853.814715,0.724138,-40441.053294,1.000000,52.986863
1,equal_1x,12,103,1.467190e+05,1424.456749,0.660194,-49261.410409,1.000000,55.626677
2,equal_1x,15,51,4.192003e+05,8219.614050,0.823529,-44768.311536,1.000000,52.702568
3,equal_1x,18,72,3.673318e+05,5101.830913,0.833333,-49418.522467,1.000000,59.122572
4,equal_1x,21,23,2.008474e+05,8732.493763,0.869565,-40497.519097,1.000000,50.278298
5,equal_1x,24,17,1.563616e+05,9197.739174,0.764706,-33174.909732,1.000000,47.593454
6,equal_1x,27,85,6.854119e+05,8063.669209,0.788235,-111511.425505,1.000000,48.045604
7,equal_1x,30,44,2.471622e+05,5617.321603,0.795455,-100415.930124,1.000000,50.746579
8,equal_1x,33,149,1.464731e+06,9830.412307,0.832215,-75007.519387,1.000000,55.068612
9,tenor_based,9,58,8.064094e+04,1390.361036,0.724138,-30330.789971,0.750000,52.986863


In [17]:
# ============================================================
# Final cell: Freeze production candidate v0.1
# ============================================================

import json

PRODUCTION_SPEC = {
    "candidate_name": "naked_atm_put_production_candidate_v0_1",
    "instrument": "naked ATM puts",
    "strike_rule": "ATM only",
    "signal_rule": "primary_with_refined_hybrid_override",
    "sizing_rule": "tenor_plus_override_sizeup",
    "tenor_size_multipliers": {
        "front_9_15d": 0.75,
        "middle_18_24d": 1.00,
        "back_27_33d": 1.15,
        "refined_hybrid_override": 1.25,
    },
    "notes": [
        "Only tenor and sizing vary.",
        "No strike optimization.",
        "No spreads.",
        "No hedge overlay.",
        "No stress engine.",
        "No portfolio caps.",
        "This is production v0.1, intended to be improved over time.",
    ],
}

PRODUCTION_SPEC_JSON = (
    AUDIT_DIR / "naked_atm_put_production_candidate_v0_1_spec.json"
)

with open(PRODUCTION_SPEC_JSON, "w") as f:
    json.dump(PRODUCTION_SPEC, f, indent=4)

print("Saved production spec:")
print(PRODUCTION_SPEC_JSON)

Saved production spec:
C:\Users\patri\vrp_project\data\audit\naked_atm_put_production_candidate_v0_1_spec.json
